In [ ]:
import torch
from PIL import Image
import os
from torchvision.transforms.v2 import PILToTensor

os.chdir("..") # to allow relative imports
from model.swinv2_encoder import *

torch.set_printoptions(linewidth=10000000, threshold=300000000)

In [ ]:
x = torch.arange(36).reshape(3, 3, 4) # B Ph*Pw C
x[0, 1, :] = -1
x[2, 2, :] = -1
x[0, 0, :] = 0
x[2, 0, :] = 0
x[2, 2, :] = 0
x

In [ ]:
sorted, abs_sorted_idxs = torch.sort(torch.abs(x), dim=1)
print(sorted, end="\n")
abs_sorted_idxs 

In [ ]:
t = torch.gather(x, 1, abs_sorted_idxs)[:, 2:]
(t == 0).all(dim=2).count_nonzero()

In [ ]:
from time import time
B = 3
num_partial_masked = 2
num_patch_specks = 3

times = []
for _ in range(1000):
    start = time()
    partial_patch_batch_idxs = torch.arange(B).repeat_interleave(
        num_partial_masked * num_patch_specks
    )
    end = time()
    times.append(end - start)

sum(times) / len(times)

In [ ]:
times = []
for _ in range(1000):
    start = time()
    partial_patch_batch_idxs = torch.arange(B).view(1, B).expand(num_partial_masked * num_patch_specks, B).contiguous().view(1, -1)
    partial_patch_batch_idxs
    end = time()
    times.append(end - start)

sum(times) / len(times)

In [ ]:
x = torch.arange(27).reshape(3, 3, 3)
x = F.pad(x, (0, 0, 0, -2, 0, 0))
x.shape

In [ ]:
from PIL import Image
from torch import int8
from torchvision.transforms.functional import pil_to_tensor, to_pil_image

im = Image.open("notebooks/NAIP_23454.jpg")
x = pil_to_tensor(im).unsqueeze(0) #.to(torch.float32)

swin = SwinTransformerV2(
    img_size=x.shape[-1],
    patch_size=16,
    in_chans=3,
    depths=[2, 2, 18, 2],
    window_size=4,
    pretrained_window_sizes=[0, 0, 0, 0],
    ape=True,
    mask_token=0,
    mask_proportion=0.,
)
masked_full_x, _, _ = swin.pre_patch_embed_mask(x)
masked_full_x = masked_full_x.squeeze(0).to(torch.uint8)
to_pil_image(masked_full_x)

In [17]:
import calendar
import pandas as pd
from datetime import datetime

df = pd.read_csv("/data/million-case/all_data.csv", sep="@")

def date_to_epoch_time(date_str):
    dt = datetime.strptime(date_str, r"%Y-%m-%d")
    return calendar.timegm(dt.timetuple())

def daytime_to_day_proportion(time_str):
    dt = datetime.strptime(time_str, r"%H:%M:%S")
    seconds_count = dt.hour * 60**2 + dt.minute * 60 + dt.second
    seconds_count_day = 24 * 60**2
    return seconds_count / seconds_count_day


df["start_date"] = df["start_date"].apply(date_to_epoch_time)
df["start_time"] = df["start_time"].apply(daytime_to_day_proportion)
df

Exception: 